# Customs Clearance Risk Engine — EDA & Statistical Testing

**Notebook 1 of 5** — Exploratory Data Analysis and Hypothesis Testing

## Objective
Surface statistically significant drivers of customs clearance holds in synthetic shipment data. This sets up feature selection and modeling decisions for the predictive notebooks that follow.

## Concepts demonstrated
- Hypothesis testing: t-test, chi-square, z-score
- p-value interpretation and the role of effect size (Cohen's d)
- Class imbalance assessment
- Correlation analysis

## Dataset
10,000 synthetic shipment records covering Apparel, Electronics, Pharma, Industrial, and Food categories across India-USA and other lanes. Generated to mirror realistic distributions seen in customs data, with documentation completeness, consignor history, and commodity type as the primary signal drivers.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Load dataset
df = pd.read_csv('../data/clearance_shipments.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (10000, 15)


,shipment_id,consignor_country,consignee_country,commodity_category,declared_value_usd,weight_kg,num_line_items,consignor_history_score,is_first_time_consignee,documentation_completeness_score,lane,hs_code,was_held_at_customs,hold_reason,clearance_time_hours
0,SHIP000000,IN,US,Industrial,312.05,9.24,1,86.0,0,63.0,IN-US,730830,0,NaN,28.3
1,SHIP000001,TH,US,Apparel,8676.47,16.26,2,100.0,0,87.0,TH-US,611048,0,NaN,5.9
2,SHIP000002,CN,US,Apparel,4908.95,7.07,1,68.0,0,84.0,CN-US,620418,0,NaN,9.6
3,SHIP000003,CN,US,Pharma,1598.98,2.20,1,59.0,0,89.0,CN-US,300486,0,NaN,7.0
4,SHIP000004,IN,US,Electronics,440.68,14.56,4,76.0,1,99.0,IN-US,851728,0,NaN,14.8


## Section 1: Basic EDA

Quick look at the data structure, types, missing values, and class balance the standard first pass before any modeling.

In [2]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   shipment_id                       10000 non-null  str    
 1   consignor_country                 10000 non-null  str    
 2   consignee_country                 10000 non-null  str    
 3   commodity_category                10000 non-null  str    
 4   declared_value_usd                10000 non-null  float64
 5   weight_kg                         10000 non-null  float64
 6   num_line_items                    10000 non-null  int64  
 7   consignor_history_score           10000 non-null  float64
 8   is_first_time_consignee           10000 non-null  int64  
 9   documentation_completeness_score  10000 non-null  float64
 10  lane                              10000 non-null  str    
 11  hs_code                           10000 non-null  int64  
 12  was_held_at_cust

In [3]:
df.describe()


,declared_value_usd,weight_kg,num_line_items,consignor_history_score,is_first_time_consignee,documentation_completeness_score,hs_code,was_held_at_customs,clearance_time_hours
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,1361.164395,12.115360,3.363700,74.373800,0.304500,81.021900,625609.682900,0.188100,22.53718
std,2401.146138,15.125697,2.791523,16.903457,0.460218,13.580363,223517.987871,0.390812,14.42956
min,3.130000,0.100000,1.000000,0.000000,0.000000,15.000000,90110.000000,0.000000,1.00000
25%,288.992500,3.720000,1.000000,63.000000,0.000000,72.000000,610955.000000,0.000000,12.90000
50%,664.310000,7.400000,2.000000,75.000000,0.000000,82.000000,620444.000000,0.000000,18.30000
75%,1499.240000,14.540000,5.000000,88.000000,1.000000,92.000000,847313.000000,0.000000,26.60000
max,58302.490000,253.800000,15.000000,100.000000,1.000000,100.000000,852898.000000,1.000000,75.00000


In [4]:
# Class balance for the binary target
balance = df['was_held_at_customs'].value_counts(normalize=True)
print("Class balance for was_held_at_customs:")
print(balance)
print(f"\nPositive class (held): {balance[1]:.1%}")
print(f"Negative class (cleared): {balance[0]:.1%}")

Class balance for was_held_at_customs:
was_held_at_customs
0    0.8119
1    0.1881
Name: proportion, dtype: float64

Positive class (held): 18.8%
Negative class (cleared): 81.2%
